In [1]:
from utils.util import *
import geopandas as gpd
from shapely.geometry import shape
import folium
import os
import sys
import time
import pandas as pd
datasets = ['landsat_ot_c2_l2', 'srtm_v3', 'ccdc_v1_3'] 
i = 0
'''['nlcd_collection_lndcov']'''
bandNames = {'B2', 'B3', 'B4', 'B5', 'B6', 'ST_B10'}
years = [2015]
includeMetadata = True

Directory './Unprocessed' already exists.
Directory './Data' already exists.
Directory './RawClippedRasters' already exists.
Directory './Data/LST' already exists.
Directory './Data/NDVI' already exists.
Directory './Data/NDWI' already exists.
Directory './Data/Land_Cover' already exists.
Directory './Data/Albedo' already exists.
Directory './Data/DEM' already exists.
Directory './RawClippedRasters/LST' already exists.
Directory './RawClippedRasters/NDVI' already exists.
Directory './RawClippedRasters/NDWI' already exists.
Directory './RawClippedRasters/Land_Cover' already exists.
Directory './RawClippedRasters/Albedo' already exists.
Directory './RawClippedRasters/DEM' already exists.
Logging in...


Login Successful, API Key Received!


In [2]:
# Cell 2: Load shape file
shapefile_folder = "./Data/area_shp/"
shapefile = "Polygon_Pahrump_NV.shp"
city = shapefile.replace('Polygon_', '').replace('.shp', '')
aoi_geodf = gpd.read_file(shapefile_folder + shapefile)
aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
if aoi_geodf.empty:
    sys.exit("Error: Shapefile contains no data.")
print("Shapefile loaded successfully.")

Shapefile loaded successfully.


In [3]:
search_payload = createSceneSearchPayload(datasets[i], aoi_geodf, years[0], 20)
scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
pd.json_normalize(scenes['results'])

,browse,cloudCover,entityId,displayId,orderingId,metadata,hasCustomizedMetadata,publishDate,options.bulk,options.download,...,options.secondary,selected.bulk,selected.compare,selected.order,spatialBounds.type,spatialBounds.coordinates,spatialCoverage.type,spatialCoverage.coordinates,temporalCoverage.endDate,temporalCoverage.startDate
0,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",0,LC80400352015052LGN01,LC08_L2SP_040035_20150221_20200909_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-117.63752, 34.97845], [-117.63752, 37.0933...",Polygon,"[[[-117.63752, 35.3716], [-115.60248, 34.97845...",2015-02-21 00:00:00,2015-02-21 00:00:00
1,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",0,LC80390352015045LGN01,LC08_L2SP_039035_20150214_20200910_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-116.07556, 34.97865], [-116.07556, 37.0936...",Polygon,"[[[-116.07556, 35.37186], [-114.04067, 34.9786...",2015-02-14 00:00:00,2015-02-14 00:00:00
2,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",16,LC80400352015036LGN01,LC08_L2SP_040035_20150205_20200909_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-117.63427, 34.9787], [-117.63427, 37.09369...",Polygon,"[[[-117.63427, 35.37195], [-115.59936, 34.9787...",2015-02-05 00:00:00,2015-02-05 00:00:00
3,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",2,LC80390352015013LGN01,LC08_L2SP_039035_20150113_20200910_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-116.08404, 34.97836], [-116.08404, 37.0934...",Polygon,"[[[-116.08404, 35.37173], [-114.04924, 34.9783...",2015-01-13 00:00:00,2015-01-13 00:00:00


In [4]:
# Cell 7: Collect Entity IDs
entityIds = [result['entityId'] for result in scenes['results'] if result['options']['bulk']]

In [5]:
# Cell 8: Prepare Scene List for Download
listId = f"temp_{datasets[i]}_list"
scn_list_add_payload = {
    "listId": listId,
    'idField': 'entityId',
    "entityIds": entityIds,
    "datasetName": datasets[i]
}
sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey)

4

In [6]:
# Cell 9: Prepare Download Options
download_opt_payload = {
    "listId": listId,
    "datasetName": datasets[i],
}
products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
pd.json_normalize(products)


,id,downloadName,displayId,entityId,datasetId,available,filesize,productName,productCode,bulkAvailable,downloadSystem,secondaryDownloads,fileGroups
0,5e83d14fec7cae84,None,LC08_L2SP_039035_20150113_20200910_02_T1,LC80390352015013LGN01,5e83d14f2fc39685,True,975547025,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
1,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_039035_20150113_20200910_02_T1,LC80390352015013LGN01,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_039035_20150113_20200910_02_T1,LC80390352015013LGN01,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
3,632210d4770592cf,None,LC08_L2SP_039035_20150113_20200910_02_T1,LC80390352015013LGN01,5e83d14f2fc39685,False,975547025,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
4,5e83d14fec7cae84,None,LC08_L2SP_039035_20150214_20200910_02_T1,LC80390352015045LGN01,5e83d14f2fc39685,True,976073199,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
5,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_039035_20150214_20200910_02_T1,LC80390352015045LGN01,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
6,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_039035_20150214_20200910_02_T1,LC80390352015045LGN01,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
7,632210d4770592cf,None,LC08_L2SP_039035_20150214_20200910_02_T1,LC80390352015045LGN01,5e83d14f2fc39685,False,976073199,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
8,5e83d14fec7cae84,None,LC08_L2SP_040035_20150205_20200909_02_T1,LC80400352015036LGN01,5e83d14f2fc39685,True,966757933,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
9,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_040035_20150205_20200909_02_T1,LC80400352015036LGN01,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None


In [7]:
# Cell 10: Collect Files to Download
downloads = []
if 'landsat_ot_c2_l2' in datasets[i]:
    for product in products:
        if product["secondaryDownloads"]:
            for secDownload in product["secondaryDownloads"]:
                if secDownload["bulkAvailable"] and any(band in secDownload['displayId'] for band in bandNames):
                    downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
                if includeMetadata and secDownload['displayId'].endswith('_MTL.txt'):
                    downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
elif 'srtm_v3' in datasets[i]:
    for product in products:
        if product["bulkAvailable"] and product["entityId"] and product["id"]:
            downloads.append({"entityId": product["entityId"], "productId": product["id"]})
elif 'ccdc_v1_3' in datasets[i]:
    for product in products:
        if product["bulkAvailable"] and product["entityId"] and product["id"]:
            downloads.append({"entityId": product["entityId"], "productId": product["id"]})

In [8]:
# Cell 11: Submit Download Request
download_req_payload = {
    "downloads": downloads,
    "label": listId
}
download_request_results = sendRequest(serviceUrl + "download-request", download_req_payload, apiKey)

In [9]:
print(download_request_results)

{'availableDownloads': [{'downloadId': 715586891, 'eulaCode': None, 'entityId': 'L2ST_LC08_L2SP_039035_20150113_20200910_02_T1_MTL_TXT', 'url': 'https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2015/039/035/LC08_L2SP_039035_20150113_20200910_02_T1/LC08_L2SP_039035_20150113_20200910_02_T1_MTL.txt?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6MjczNjI1NzcsImRvd25sb2FkSWQiOjcxNTU4Njg5MSwiZGF0ZUdlbmVyYXRlZCI6IjIwMjQtMTItMjFUMTM6NTU6MDctMDY6MDAiLCJpZCI6IkxDMDhfTDJTUF8wMzkwMzVfMjAxNTAxMTNfMjAyMDA5MTBfMDJfVDFfTVRMLnR4dCIsInNpZ25hdHVyZSI6IiQ1JCRtOXAzS3R4RFJkUzhLbmg5N2JYbEpHTVJuQllZWlp4d0RoR0sxb2JvbEZBIn0='}, {'downloadId': 715586892, 'eulaCode': None, 'entityId': 'L2SR_LC08_L2SP_039035_20150113_20200910_02_T1_SR_B2_TIF', 'url': 'https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2015/039/035/LC08_L2SP_039035_20150113_20200910_02_T1/LC08_L2SP_039035_20150113_20200910_02_T1_SR_B2.TIF?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRh

In [10]:
# Cell 12: Download Files
if datasets[i] == 'landsat_ot_c2_l2':
    results = download_request_results['availableDownloads']
    for result in results:
        runDownload(threads, result['url'])
elif datasets[i] == 'srtm_v3':
    result = download_request_results['preparingDownloads'][2]
    runDownload(threads, result['url'])
else:
    result = download_request_results['preparingDownloads'][0]
    runDownload(threads, result['url'])
for _, t in enumerate(threads):
    t.join()
for file in os.listdir('./Unprocessed'):
    if file.endswith('.tar'):
        try: 
            extract_specific_files('./Unprocessed/' + file, './Unprocessed')
        except:
            print(f'Error: Could not extract file {file}')

    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_MTL.txt...
    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_SR_B4.TIF...
    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_SR_B2.TIF...
    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_SR_B5.TIF...
    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_SR_B6.TIF...
    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_SR_B3.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_MTL.txt...
    Downloading: LC08_L2SP_039035_20150113_20200910_02_T1_ST_B10.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_SR_B2.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_SR_B3.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_SR_B4.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_SR_B5.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_SR_B6.TIF...
    Downloading: LC08_L2SP_039035_20150214_20200910_02_T1_ST_B10.TIF...
    Down

In [11]:
# Cell 13: Clean Up
remove_scnlst_payload = {"listId": listId}
sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)

In [12]:
# Cell 15: Verify Downloads
import rasterio
from shapely.geometry import box

usableTIFFs = []
print(os.listdir(unprocessed_dir))
# Ensure the shapefile matches the TIF CRS
for tif_file in os.listdir(unprocessed_dir):
    if tif_file.endswith(".TIF") or tif_file.endswith(".tif"):
        tif_path = unprocessed_dir + '/' + tif_file
        with rasterio.open(tif_path) as src:
            # Reproject shapefile to match TIF file CRS
            aoi_geodf_proj = aoi_geodf.to_crs(src.crs)
            tif_bounds = box(*src.bounds)
            
            # Check containment
            for idx, geom in enumerate(aoi_geodf_proj.geometry):
                if tif_bounds.contains(geom):
                    usableTIFFs.append(tif_file)
                    print(f"Polygon {city} is fully inside {tif_file}")
                else:
                    print(f"Polygon {city} is NOT fully inside {tif_file}")
print(usableTIFFs)

['LC08_L2SP_039035_20150113_20200910_02_T1_MTL.txt', 'LC08_L2SP_039035_20150113_20200910_02_T1_SR_B2.TIF', 'LC08_L2SP_039035_20150113_20200910_02_T1_SR_B3.TIF', 'LC08_L2SP_039035_20150113_20200910_02_T1_SR_B4.TIF', 'LC08_L2SP_039035_20150113_20200910_02_T1_SR_B5.TIF', 'LC08_L2SP_039035_20150113_20200910_02_T1_SR_B6.TIF', 'LC08_L2SP_039035_20150113_20200910_02_T1_ST_B10.TIF', 'LC08_L2SP_039035_20150214_20200910_02_T1_MTL.txt', 'LC08_L2SP_039035_20150214_20200910_02_T1_SR_B2.TIF', 'LC08_L2SP_039035_20150214_20200910_02_T1_SR_B3.TIF', 'LC08_L2SP_039035_20150214_20200910_02_T1_SR_B4.TIF', 'LC08_L2SP_039035_20150214_20200910_02_T1_SR_B5.TIF', 'LC08_L2SP_039035_20150214_20200910_02_T1_SR_B6.TIF', 'LC08_L2SP_039035_20150214_20200910_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20150205_20200909_02_T1_MTL.txt', 'LC08_L2SP_040035_20150205_20200909_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20150205_20200909_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20150205_20200909_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20150205_2020

In [ ]:
goodCoordinates = clipUnprocessedRasters(usableTIFFs, aoi_geodf_proj)

Clipped, reprojected, and color-preserved TIF saved as ./Unprocessed/Clipped_LC08_L2SP_040035_20150205_20200909_02_T1_SR_B2.TIF
Clipped, reprojected, and color-preserved TIF saved as ./Unprocessed/Clipped_LC08_L2SP_040035_20150205_20200909_02_T1_SR_B3.TIF
Clipped, reprojected, and color-preserved TIF saved as ./Unprocessed/Clipped_LC08_L2SP_040035_20150205_20200909_02_T1_SR_B4.TIF
Clipped, reprojected, and color-preserved TIF saved as ./Unprocessed/Clipped_LC08_L2SP_040035_20150205_20200909_02_T1_SR_B5.TIF


In [ ]:
for file in os.listdir(unprocessed_dir):
    if ".txt" in file or "Clipped_" in file:
        if '1arc_v3' in file:
            moveToRaw(file, 'DEM', f'{years[0]}-01-01', city)
            continue
        if 'LCMAP' in file:
            if 'LCPRI' in file:
                moveToRaw(file, 'Land_Cover', f'{years[0]}-01-01', city)
            continue
        date, band, coordinate = getMetaFromLandsatTIRs(file)
        if coordinate not in goodCoordinates:
            continue
        print(date, band)
        if band == 'B10':
            moveToRaw(file, 'LST', date, city)
        if band == 'B2':
            moveToRaw(file, 'Albedo', date, city)
        if band == 'B3':
            moveToRaw(file, 'NDWI', date, city)
        if band == 'B4':
            moveToRaw(file, 'Albedo', date, city)
            moveToRaw(file, 'NDVI', date, city)
        if band == 'B5':
            moveToRaw(file, 'Albedo', date, city)
            moveToRaw(file, 'NDVI', date, city)
            moveToRaw(file, 'NDWI', date, city)
        if band == 'B6':
            moveToRaw(file, 'Albedo', date, city)
        if band == 'MTL':
            moveToRaw(file, 'Albedo', date, city)
            moveToRaw(file, 'LST', date, city)
            moveToRaw(file, 'NDVI', date, city)
            moveToRaw(file, 'NDWI', date, city)    

In [ ]:
clear_folder(unprocessed_dir)